# Edge-Centric Network Analysis with igraph

## Overview

This notebook implements **edge-centric network analysis** where:
- Each connection (edge) in your network becomes a node in a new network
- Two edges are connected if they share a node (line graph transformation)
- Reveals higher-order organizational principles not visible in traditional node-level analysis
---

## 1. Setup and Installation

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import igraph as ig
from scipy.stats import entropy, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully")
print(f"igraph version: {ig.__version__}")

---
## 2. Load Connectivity Data


In [ ]:
df = pd.read_csv('../data/w_ij_gaa.csv', index_col=0)

# Extract connectivity matrix and labels
W = df.values
node_labels = df.columns.tolist()

print(f"Loaded connectivity matrix: {W.shape[0]} nodes")
print(f"Matrix shape: {W.shape}")
print(f"\nFirst few node labels:")
for i, label in enumerate(node_labels[:5]):
    print(f"  {i}: {label}")

# Display the connectivity matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(W, cmap='viridis', aspect='auto')
ax.set_title('Original Connectivity Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Target Node')
ax.set_ylabel('Source Node')
plt.colorbar(im, ax=ax, label='Connection Weight')
plt.tight_layout()
plt.show()

---
## 3. Build Node-Level Network

First, create a traditional node-level network using igraph.

In [ ]:
# Configuration
DIRECTED = True  # Set to False for undirected networks

# Extract edges from connectivity matrix
edges_list = []
edge_weights = []

n_nodes = W.shape[0]

if DIRECTED:
    # For directed networks: include all i->j connections
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j and not np.isnan(W[i,j]) and W[i,j] != 0:
                edges_list.append((i, j))
                edge_weights.append(W[i,j])
else:
    # For undirected networks: only upper triangle
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if not np.isnan(W[i,j]) and W[i,j] != 0:
                edges_list.append((i, j))
                edge_weights.append(W[i,j])

# Create node-level igraph network
G_nodes = ig.Graph(directed=DIRECTED)
G_nodes.add_vertices(n_nodes)
G_nodes.vs['name'] = node_labels
G_nodes.add_edges(edges_list)
G_nodes.es['weight'] = edge_weights

print("="*60)
print("NODE-LEVEL NETWORK")
print("="*60)
print(f"Nodes: {G_nodes.vcount()}")
print(f"Edges: {G_nodes.ecount()}")
print(f"Directed: {DIRECTED}")
print(f"Density: {G_nodes.density():.4f}")
print(f"Connected components: {len(G_nodes.components())}")

---
## 4. Build Edge-Level Network (Line Graph Transformation)

Transform the node network into an edge network where:
- Each edge becomes a node
- Two edges are connected if they share a node

In [ ]:
# Extract edge information from node network
edge_nodes = []  # (source, target) for each edge
edge_labels = []  # String labels for each edge
edge_weights_list = []  # Original weights

for e in G_nodes.es:
    source = e.source
    target = e.target
    edge_nodes.append((source, target))
    
    # Create label
    source_name = G_nodes.vs[source]['name']
    target_name = G_nodes.vs[target]['name']
    arrow = "→" if DIRECTED else "-"
    edge_labels.append(f"{source_name}{arrow}{target_name}")
    
    edge_weights_list.append(e['weight'])

n_edge_nodes = len(edge_nodes)
print(f"Creating edge network with {n_edge_nodes} edge nodes...")

# Build edge-to-edge connections
# Two edges are connected if they share a node
edge_connections = []
connection_types = []

for i in range(n_edge_nodes):
    if i % 100 == 0:
        print(f"  Processing edge {i}/{n_edge_nodes}...", end='\r')
    
    for j in range(i+1, n_edge_nodes):
        edge1 = edge_nodes[i]  # (source1, target1)
        edge2 = edge_nodes[j]  # (source2, target2)
        
        connected = False
        conn_type = ""
        
        if DIRECTED:
            # Check multiple connection types for directed networks
            if edge1[1] == edge2[0]:  # target1 = source2 (sequential)
                connected = True
                conn_type = "sequential"
            elif edge1[0] == edge2[0]:  # same source
                connected = True
                conn_type = "common_source"
            elif edge1[1] == edge2[1]:  # same target
                connected = True
                conn_type = "common_target"
            elif edge1[0] == edge2[1]:  # source1 = target2 (reverse)
                connected = True
                conn_type = "reverse_sequential"
        else:
            # For undirected: edges share a node
            if len(set(edge1) & set(edge2)) > 0:
                connected = True
                conn_type = "shared_node"
        
        if connected:
            edge_connections.append((i, j))
            connection_types.append(conn_type)

print(f"  Found {len(edge_connections)} edge-edge connections")

# Create edge-level igraph network
G_edges = ig.Graph(directed=False)  # Edge network is typically undirected
G_edges.add_vertices(n_edge_nodes)
G_edges.vs['name'] = edge_labels
G_edges.vs['original_weight'] = edge_weights_list
G_edges.vs['source_node'] = [e[0] for e in edge_nodes]
G_edges.vs['target_node'] = [e[1] for e in edge_nodes]
G_edges.add_edges(edge_connections)
G_edges.es['connection_type'] = connection_types

print("\n" + "="*60)
print("EDGE-LEVEL NETWORK")
print("="*60)
print(f"Edge nodes: {G_edges.vcount()}")
print(f"Edge-edge connections: {G_edges.ecount()}")
print(f"Average degree: {np.mean(G_edges.degree()):.2f}")
print(f"Density: {G_edges.density():.4f}")
print(f"Connected components: {len(G_edges.components())}")

# Connection type distribution
if DIRECTED:
    type_counts = pd.Series(connection_types).value_counts()
    print("\nConnection type distribution:")
    for conn_type, count in type_counts.items():
        print(f"  {conn_type}: {count} ({100*count/len(connection_types):.1f}%)")

---
## 5. Create Edge Functional Connectivity (eFC) Matrix

The eFC matrix is the adjacency matrix of the edge network.

In [ ]:
# Get adjacency matrix (edge FC matrix)
eFC = np.array(G_edges.get_adjacency().data)

print("="*60)
print("EDGE FUNCTIONAL CONNECTIVITY MATRIX")
print("="*60)
print(f"Shape: {eFC.shape}")
print(f"Total possible connections: {eFC.shape[0] * eFC.shape[1]}")
print(f"Actual connections: {np.sum(eFC > 0)}")
print(f"Sparsity: {1 - np.sum(eFC > 0)/(eFC.shape[0] * eFC.shape[1]):.4f}")

# Visualize eFC matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Original connectivity
ax = axes[0]
im = ax.imshow(W, cmap='viridis', aspect='auto')
ax.set_title('Original Connectivity Matrix\n'
             f'({n_nodes} × {n_nodes})', fontsize=12, fontweight='bold')
ax.set_xlabel('Target Node')
ax.set_ylabel('Source Node')
plt.colorbar(im, ax=ax, label='Weight')

# Edge FC matrix
ax = axes[1]
im = ax.imshow(eFC, cmap='binary', aspect='auto')
ax.set_title(f'Edge Functional Connectivity Matrix\n'
             f'({eFC.shape[0]} × {eFC.shape[1]})', fontsize=12, fontweight='bold')
ax.set_xlabel('Edge Index')
ax.set_ylabel('Edge Index')
plt.colorbar(im, ax=ax, label='Connected')

plt.tight_layout()
plt.show()

---
## 6. Compute Centrality Measures

Calculate various centrality measures for edges in the edge network.

In [ ]:
print("Computing centrality measures...")

# Calculate centralities
degree_centrality = np.array(G_edges.degree())
betweenness_centrality = np.array(G_edges.betweenness())
closeness_centrality = np.array(G_edges.closeness())
eigenvector_centrality = np.array(G_edges.eigenvector_centrality())
pagerank_centrality = np.array(G_edges.pagerank())

# Add to graph attributes
G_edges.vs['degree'] = degree_centrality.tolist()
G_edges.vs['betweenness'] = betweenness_centrality.tolist()
G_edges.vs['closeness'] = closeness_centrality.tolist()
G_edges.vs['eigenvector'] = eigenvector_centrality.tolist()
G_edges.vs['pagerank'] = pagerank_centrality.tolist()

print("✓ Centrality measures computed")

# Summary statistics
print("\n" + "="*60)
print("CENTRALITY STATISTICS")
print("="*60)
print(f"Degree:      mean={degree_centrality.mean():.2f}, max={degree_centrality.max():.0f}")
print(f"Betweenness: mean={betweenness_centrality.mean():.2f}, max={betweenness_centrality.max():.2f}")
print(f"Closeness:   mean={closeness_centrality.mean():.4f}, max={closeness_centrality.max():.4f}")
print(f"Eigenvector: mean={eigenvector_centrality.mean():.4f}, max={eigenvector_centrality.max():.4f}")
print(f"PageRank:    mean={pagerank_centrality.mean():.6f}, max={pagerank_centrality.max():.6f}")

# Find most central edges
top_k = 10
print(f"\n{'='*60}")
print(f"TOP {top_k} EDGES BY BETWEENNESS CENTRALITY")
print("="*60)
top_idx = np.argsort(betweenness_centrality)[-top_k:]
for rank, idx in enumerate(reversed(top_idx), 1):
    print(f"{rank:2d}. {edge_labels[idx][:60]}")
    print(f"    Betweenness: {betweenness_centrality[idx]:.2f}, "
          f"Degree: {degree_centrality[idx]:.0f}, "
          f"Weight: {edge_weights_list[idx]:.2f}")

### Visualize Centrality Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

centralities = [
    ('Degree', degree_centrality),
    ('Betweenness', betweenness_centrality),
    ('Closeness', closeness_centrality),
    ('Eigenvector', eigenvector_centrality),
    ('PageRank', pagerank_centrality)
]

for idx, (name, values) in enumerate(centralities):
    ax = axes[idx // 3, idx % 3]
    ax.hist(values, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(values), color='red', linestyle='--', 
               linewidth=2, label=f'Mean={np.mean(values):.2e}')
    ax.set_xlabel(name)
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name} Distribution')
    ax.legend()
    ax.grid(alpha=0.3)

# Remove empty subplot
axes[1, 2].remove()

plt.tight_layout()
plt.show()

---
## 7. Community Detection

Detect communities in the edge network. Edge communities can reveal overlapping functional modules.

In [ ]:
# Try multiple community detection methods
print("Testing community detection methods...\n")

methods = ['multilevel', 'leiden', 'infomap', 'label_propagation']
results = {}

for method in methods:
    try:
        if method == 'multilevel':
            communities = G_edges.community_multilevel()
        elif method == 'leiden':
            communities = G_edges.community_leiden()
        elif method == 'infomap':
            communities = G_edges.community_infomap()
        elif method == 'label_propagation':
            communities = G_edges.community_label_propagation()
        
        results[method] = communities
        print(f"{method:20s}: {len(communities):3d} communities, "
              f"Q={communities.modularity:.4f}, "
              f"sizes={[len(c) for c in communities][:5]}...")
    except Exception as e:
        print(f"{method:20s}: Not available - {e}")

# Use the method with highest modularity
best_method = max(results.items(), key=lambda x: x[1].modularity)
communities = best_method[1]

print(f"\n✓ Using {best_method[0]} (highest modularity)")

# Add community assignments to graph
G_edges.vs['community'] = communities.membership

print("\n" + "="*60)
print("COMMUNITY DETECTION RESULTS")
print("="*60)
print(f"Method: {best_method[0]}")
print(f"Number of communities: {len(communities)}")
print(f"Modularity: {communities.modularity:.4f}")
print(f"Community sizes: {[len(c) for c in communities]}")

### Analyze Community Structure

In [ ]:
# Analyze which original nodes are involved in each edge community
print("\n" + "="*60)
print("EDGE COMMUNITIES → ORIGINAL NODES")
print("="*60)

for comm_idx, community in enumerate(communities):
    if len(community) < 3:  # Skip very small communities
        continue
    
    # Get all original nodes involved in this edge community
    nodes_in_community = set()
    for edge_idx in community:
        source = edge_nodes[edge_idx][0]
        target = edge_nodes[edge_idx][1]
        nodes_in_community.add(source)
        nodes_in_community.add(target)
    
    print(f"\nCommunity {comm_idx} ({len(community)} edges, "
          f"{len(nodes_in_community)} unique nodes):")
    
    # Sample nodes
    node_names = [node_labels[n] for n in list(nodes_in_community)[:8]]
    print(f"  Sample nodes: {', '.join(node_names[:5])}")
    if len(node_names) > 5:
        print(f"                {', '.join(node_names[5:8])}...")
    
    # Most connected edges in this community
    edges_in_comm = [(edge_labels[i], degree_centrality[i]) for i in community]
    edges_in_comm.sort(key=lambda x: x[1], reverse=True)
    print(f"  Most connected edge: {edges_in_comm[0][0][:55]}")
    print(f"                       (degree={edges_in_comm[0][1]:.0f})")

### Visualize Edge FC Matrix Ordered by Communities

In [ ]:
# Reorder matrix by communities
community_order = np.argsort(communities.membership)
eFC_reordered = eFC[community_order, :][:, community_order]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Original order
ax = axes[0]
im = ax.imshow(eFC, cmap='binary', aspect='auto')
ax.set_title('Edge FC Matrix (original order)', fontsize=12, fontweight='bold')
ax.set_xlabel('Edge Index')
ax.set_ylabel('Edge Index')
plt.colorbar(im, ax=ax, label='Connected')

# Ordered by communities
ax = axes[1]
im = ax.imshow(eFC_reordered, cmap='binary', aspect='auto')
ax.set_title(f'Edge FC Matrix (ordered by {len(communities)} communities)', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Edge Index')
ax.set_ylabel('Edge Index')

# Add community boundaries
boundaries = [0]
for c in communities:
    boundaries.append(boundaries[-1] + len(c))
for b in boundaries[1:-1]:
    ax.axhline(b, color='red', linewidth=1, alpha=0.6)
    ax.axvline(b, color='red', linewidth=1, alpha=0.6)

plt.colorbar(im, ax=ax, label='Connected')
plt.tight_layout()
plt.show()

print(f"✓ Communities show block-diagonal structure (modularity = {communities.modularity:.4f})")

---
## 8. Project Communities to Nodes (Overlapping Communities)

Edge communities can be projected back onto nodes, creating overlapping node communities.

In [ ]:
# Calculate node participation in each edge community
n_communities = len(communities)
participation = np.zeros((n_nodes, n_communities))

# Count how many edges in each community connect to each node
for edge_idx, (source, target) in enumerate(edge_nodes):
    comm = communities.membership[edge_idx]
    participation[source, comm] += 1
    participation[target, comm] += 1

# Normalize by total edges per node
row_sums = participation.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # Avoid division by zero
participation = participation / row_sums

print("="*60)
print("NODE PARTICIPATION IN EDGE COMMUNITIES")
print("="*60)
print(f"Participation matrix shape: {participation.shape}")
print(f"(nodes × communities)")

# Calculate overlap (entropy)
node_entropy = np.array([entropy(p) for p in participation])

# Nodes with highest overlap
top_overlap = 10
high_overlap_idx = np.argsort(node_entropy)[-top_overlap:]

print(f"\nTop {top_overlap} nodes with highest community overlap:")
for rank, idx in enumerate(reversed(high_overlap_idx), 1):
    # Get top communities for this node
    top_comms = np.argsort(participation[idx])[-3:]
    comm_str = ", ".join([f"C{c}({participation[idx,c]:.2f})" for c in reversed(top_comms)])
    print(f"{rank:2d}. {node_labels[idx]:40s} entropy={node_entropy[idx]:.3f} [{comm_str}]")

# Visualize participation matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Participation matrix
ax = axes[0]
im = ax.imshow(participation, cmap='YlOrRd', aspect='auto')
ax.set_title('Node Participation in Edge Communities', fontsize=12, fontweight='bold')
ax.set_xlabel('Community')
ax.set_ylabel('Node')
plt.colorbar(im, ax=ax, label='Participation')

# Node entropy distribution
ax = axes[1]
ax.hist(node_entropy, bins=30, edgecolor='black', alpha=0.7, color='coral')
ax.axvline(node_entropy.mean(), color='red', linestyle='--', 
           linewidth=2, label=f'Mean={node_entropy.mean():.3f}')
ax.set_xlabel('Entropy (overlap)')
ax.set_ylabel('Number of Nodes')
ax.set_title('Distribution of Node Community Overlap', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Network Visualization

Visualize the edge network (using largest connected component for clarity).

In [ ]:
# Get largest component for visualization
components = G_edges.components()
largest_comp = max(components, key=len)
G_vis = G_edges.subgraph(largest_comp)

print(f"Visualizing largest component: {len(largest_comp)} / {G_edges.vcount()} nodes")

# Calculate layout (this may take a moment for large networks)
print("Computing network layout...")
layout = G_vis.layout_fruchterman_reingold()
print("✓ Layout computed")

# Prepare visualization attributes
colors_community = [communities.membership[i] for i in largest_comp]
sizes_betweenness = [G_vis.vs[i]['betweenness'] for i in range(len(G_vis.vs))]
max_between = max(sizes_betweenness) if max(sizes_betweenness) > 0 else 1
vertex_sizes = [5 + 25 * (b / max_between) for b in sizes_betweenness]

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Plot 1: Colored by community
ax = axes[0]
ig.plot(
    G_vis,
    target=ax,
    layout=layout,
    vertex_size=12,
    vertex_color=colors_community,
    vertex_frame_width=0.5,
    edge_width=0.3,
    edge_color='#CCCCCC',
    palette=ig.RainbowPalette(n=len(communities))
)
ax.set_title(f'Edge Network - Colored by Community\n'
             f'(Largest Component: {len(largest_comp)} edges, '
             f'{len(communities)} communities)',
             fontsize=12, fontweight='bold')
ax.axis('off')

# Plot 2: Sized by betweenness centrality
ax = axes[1]

# Label high-importance edges
betweenness_threshold = np.percentile(sizes_betweenness, 95)
labels = [G_vis.vs[i]['name'][:25] if sizes_betweenness[i] > betweenness_threshold 
          else '' for i in range(len(G_vis.vs))]

ig.plot(
    G_vis,
    target=ax,
    layout=layout,
    vertex_size=vertex_sizes,
    vertex_color='lightblue',
    vertex_frame_color='navy',
    vertex_frame_width=0.5,
    vertex_label=labels,
    vertex_label_size=7,
    edge_width=0.3,
    edge_color='#CCCCCC'
)
ax.set_title(f'Edge Network - Sized by Betweenness Centrality\n'
             f'(Largest Component: {len(largest_comp)} edges)',
             fontsize=12, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Network visualization complete")

---
## 10. Summary Statistics

Create a comprehensive DataFrame with all edge statistics.

In [ ]:
# Create comprehensive statistics DataFrame
stats_df = pd.DataFrame({
    'edge_label': edge_labels,
    'source_node': [node_labels[e[0]] for e in edge_nodes],
    'target_node': [node_labels[e[1]] for e in edge_nodes],
    'source_idx': [e[0] for e in edge_nodes],
    'target_idx': [e[1] for e in edge_nodes],
    'original_weight': edge_weights_list,
    'degree': degree_centrality,
    'betweenness': betweenness_centrality,
    'closeness': closeness_centrality,
    'eigenvector': eigenvector_centrality,
    'pagerank': pagerank_centrality,
    'community': communities.membership
})

# Sort by betweenness
stats_df = stats_df.sort_values('betweenness', ascending=False)

print("="*60)
print("EDGE STATISTICS SUMMARY")
print("="*60)
print(f"Total edges analyzed: {len(stats_df)}")
print(f"\nTop 20 edges by betweenness centrality:")
print(stats_df.head(20)[['edge_label', 'betweenness', 'degree', 'community']])

# Save to file (optional)
# stats_df.to_csv('edge_statistics.csv', index=False)
# print("\n✓ Statistics saved to 'edge_statistics.csv'")

---
## 11. Advanced Analyses

### 11.1 Hub Edge Analysis

In [ ]:
# Define hubs as edges with both high degree AND high betweenness
degree_threshold = stats_df['degree'].quantile(0.90)
betweenness_threshold = stats_df['betweenness'].quantile(0.90)

hub_edges = stats_df[
    (stats_df['degree'] > degree_threshold) & 
    (stats_df['betweenness'] > betweenness_threshold)
]

print("="*60)
print("HUB EDGE ANALYSIS")
print("="*60)
print(f"Hub definition: top 10% in BOTH degree AND betweenness")
print(f"Number of hub edges: {len(hub_edges)}")
print(f"\nHub edges:")
print(hub_edges[['edge_label', 'degree', 'betweenness', 'community']].head(15))

# Visualize hubs
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(stats_df['degree'], stats_df['betweenness'], 
           alpha=0.5, s=20, label='All edges')
ax.scatter(hub_edges['degree'], hub_edges['betweenness'], 
           alpha=0.8, s=100, c='red', marker='*', label='Hub edges')
ax.axvline(degree_threshold, color='red', linestyle='--', alpha=0.5)
ax.axhline(betweenness_threshold, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Degree', fontsize=12)
ax.set_ylabel('Betweenness Centrality', fontsize=12)
ax.set_title('Hub Edge Identification', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 11.2 Weight vs Centrality Correlation

In [ ]:
# Test correlation between original edge weight and edge network centrality
corr_deg, p_deg = spearmanr(stats_df['original_weight'], stats_df['degree'])
corr_bet, p_bet = spearmanr(stats_df['original_weight'], stats_df['betweenness'])

print("="*60)
print("WEIGHT VS CENTRALITY CORRELATION")
print("="*60)
print(f"Original weight vs degree:      r={corr_deg:+.3f}, p={p_deg:.3e}")
print(f"Original weight vs betweenness: r={corr_bet:+.3f}, p={p_bet:.3e}")
print("\nInterpretation:")
if abs(corr_deg) < 0.3:
    print("  • Weak correlation suggests edge topology != original weight")
elif abs(corr_deg) < 0.7:
    print("  • Moderate correlation suggests some relationship")
else:
    print("  • Strong correlation suggests edge topology reflects original weight")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(stats_df['original_weight'], stats_df['degree'], alpha=0.5, s=20)
ax.set_xlabel('Original Edge Weight', fontsize=11)
ax.set_ylabel('Edge Degree', fontsize=11)
ax.set_title(f'Weight vs Degree\n(r={corr_deg:.3f}, p={p_deg:.3e})', 
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(stats_df['original_weight'], stats_df['betweenness'], alpha=0.5, s=20)
ax.set_xlabel('Original Edge Weight', fontsize=11)
ax.set_ylabel('Edge Betweenness', fontsize=11)
ax.set_title(f'Weight vs Betweenness\n(r={corr_bet:.3f}, p={p_bet:.3e})', 
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 11.3 Community Composition Analysis

In [ ]:
# Analyze which nodes contribute most to each community
print("="*60)
print("COMMUNITY COMPOSITION ANALYSIS")
print("="*60)

for comm_idx in range(len(communities)):
    # Get edges in this community
    comm_edges = [i for i, c in enumerate(communities.membership) if c == comm_idx]
    
    if len(comm_edges) < 3:
        continue
    
    # Count node participation
    node_counts = {}
    for edge_idx in comm_edges:
        source, target = edge_nodes[edge_idx]
        node_counts[source] = node_counts.get(source, 0) + 1
        node_counts[target] = node_counts.get(target, 0) + 1
    
    # Top contributing nodes
    top_nodes = sorted(node_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    
    print(f"\nCommunity {comm_idx} ({len(comm_edges)} edges):")
    print(f"  Top participating nodes:")
    for node_idx, count in top_nodes:
        print(f"    {node_labels[node_idx]:40s} ({count} edges)")

---
## 12. Export Results

Save all results for further analysis or sharing.

In [ ]:
# 1. Save edge statistics
stats_df.to_csv('edge_statistics.csv', index=False)
print("✓ Saved: edge_statistics.csv")

# 2. Save node participation matrix
participation_df = pd.DataFrame(
    participation,
    index=node_labels,
    columns=[f"Community_{i}" for i in range(participation.shape[1])]
)
participation_df.to_csv('node_community_participation.csv')
print("✓ Saved: node_community_participation.csv")

# 3. Save edge FC matrix
np.save('edge_fc_matrix.npy', eFC)
print("✓ Saved: edge_fc_matrix.npy")

# 4. Export networks for external tools
G_edges.write_graphml('edge_network.graphml')
print("✓ Saved: edge_network.graphml (for Gephi, Cytoscape, etc.)")

# 5. Save summary report
with open('analysis_summary.txt', 'w') as f:
    f.write("="*60 + "\n")
    f.write("EDGE-CENTRIC NETWORK ANALYSIS SUMMARY\n")
    f.write("="*60 + "\n\n")
    f.write(f"Node-level network:\n")
    f.write(f"  Nodes: {G_nodes.vcount()}\n")
    f.write(f"  Edges: {G_nodes.ecount()}\n")
    f.write(f"  Directed: {DIRECTED}\n\n")
    f.write(f"Edge-level network:\n")
    f.write(f"  Edge nodes: {G_edges.vcount()}\n")
    f.write(f"  Edge-edge connections: {G_edges.ecount()}\n")
    f.write(f"  Average degree: {np.mean(G_edges.degree()):.2f}\n")
    f.write(f"  Density: {G_edges.density():.4f}\n\n")
    f.write(f"Community structure:\n")
    f.write(f"  Method: {best_method[0]}\n")
    f.write(f"  Number of communities: {len(communities)}\n")
    f.write(f"  Modularity: {communities.modularity:.4f}\n")
    f.write(f"  Community sizes: {[len(c) for c in communities]}\n")

print("✓ Saved: analysis_summary.txt")

print("\n" + "="*60)
print("ALL RESULTS EXPORTED SUCCESSFULLY")
print("="*60)

---
## 13. Final Summary and Next Steps

In [ ]:
print("="*60)
print("EDGE-CENTRIC NETWORK ANALYSIS COMPLETE")
print("="*60)
print("\nKey Findings:")
print(f"  • Transformed {G_nodes.vcount()} nodes → {G_edges.vcount()} edge nodes")
print(f"  • Detected {len(communities)} overlapping edge communities")
print(f"  • Modularity: {communities.modularity:.4f}")
print(f"  • Identified {len(hub_edges)} hub edges")
print(f"  • Average node community overlap: {node_entropy.mean():.3f}")

print("\nGenerated Files:")
print("  • edge_statistics.csv - Complete edge metrics")
print("  • node_community_participation.csv - Overlapping communities")
print("  • edge_fc_matrix.npy - Edge connectivity matrix")
print("  • edge_network.graphml - Network file for visualization tools")
print("  • analysis_summary.txt - Text summary")

print("\nNext Steps:")
print("  1. Explore edge_statistics.csv to find interesting patterns")
print("  2. Use node_community_participation.csv for functional analysis")
print("  3. Import edge_network.graphml into Gephi/Cytoscape for interactive viz")
print("  4. Compare edge communities with known functional systems")
print("  5. Investigate hub edges for potential intervention targets")

print("\n" + "="*60)
print("Thank you for using Edge-Centric Network Analysis!")
print("="*60)
print("\nCitation:")
print("Betzel, R. F., Faskowitz, J., & Sporns, O. (2023).")
print("Living on the edge: network neuroscience beyond nodes.")
print("Trends in Cognitive Sciences, 27(11), 1068-1084.")